# PS 47 – Daily Journal Summarizer (run in Google Colab)
**Raginee Chunkhare · Roll No 47 · LY-AIA-7**

Run the cells top to bottom. Your Groq key is read from Colab **Secrets** (🔑 icon in the left bar) — it is never written into this notebook.

## 1. Get the project files into Colab
Option A: click the cell and upload `app.py`, `journal_llm.py`, `sample_journal.csv`, `requirements.txt`.

Option B (after you push to GitHub): comment out Option A and uncomment the `git clone` lines.

In [ ]:
# Option A - upload files
from google.colab import files
uploaded = files.upload()

# Option B - clone from GitHub (replace <username>)
# !git clone https://github.com/<username>/llm-ps47-ADT23SOCB0826.git
# %cd llm-ps47-ADT23SOCB0826

!ls

# Pink-purple theme file (Colab upload can't create the hidden .streamlit folder)
os_cfg = '[theme]\nbase = "light"\nprimaryColor = "#C04BC9"\nbackgroundColor = "#FFF5FB"\nsecondaryBackgroundColor = "#F3E4FF"\ntextColor = "#4A2C5E"\n'
import os; os.makedirs('.streamlit', exist_ok=True); open('.streamlit/config.toml', 'w').write(os_cfg)

## 2. Install libraries

In [ ]:
!pip install -q -r requirements.txt

## 3. Load your Groq API key from Colab Secrets
In the 🔑 Secrets panel, your key must have **Notebook access** switched ON. If your secret has a different name, add it to the list below.

In [ ]:
import os
from google.colab import userdata

GROQ_API_KEY = None
for name in ['GROQ_API_KEY', 'GROQ_KEY', 'groq_api_key', 'groq_key', 'groq', 'GROQ']:
    try:
        GROQ_API_KEY = userdata.get(name)
        print(f'Found key in secret: {name}')
        break
    except Exception:
        pass

assert GROQ_API_KEY, 'No Groq key found - check the secret name and that Notebook access is ON'
os.environ['GROQ_API_KEY'] = GROQ_API_KEY   # the Streamlit app reads it from here

## 4. Test the prompt first ("playground" step)
Tags 3 sample entries and writes one weekly summary, so you can check the outputs before starting the UI. Screenshot this output for your report.

In [ ]:
import pandas as pd, json
from journal_llm import get_client, tag_entry, summarize_week, DEFAULT_MODEL

client = get_client(os.environ['GROQ_API_KEY'])
df = pd.read_csv('sample_journal.csv', parse_dates=['date']).head(7)

for i, row in df.head(3).iterrows():
    tags = tag_entry(client, DEFAULT_MODEL, str(row['date'].date()), row['entry'])
    print(row['date'].date(), '->', json.dumps(tags, indent=1))

for col in ['mood', 'mood_score']:
    df[col] = None
for i, row in df.iterrows():
    t = tag_entry(client, DEFAULT_MODEL, str(row['date'].date()), row['entry'])
    df.at[i, 'mood'], df.at[i, 'mood_score'] = t['mood'], t['mood_score']

print('\nWEEKLY SUMMARY:')
print(json.dumps(summarize_week(client, DEFAULT_MODEL, df), indent=2))

## 5. Start the Streamlit app and open it through a public link
Wait ~15 seconds; click the `https://....trycloudflare.com` link that gets printed. Keep this cell running while you use the app. Stop it with the ■ button when done.

In [ ]:
import subprocess, time, re

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

st_proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                           stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(5)
cf_proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in cf_proc.stdout:
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        print('\n👉 OPEN YOUR APP HERE:', m.group(0), '\n')
        break

try:
    cf_proc.wait()
except KeyboardInterrupt:
    cf_proc.terminate(); st_proc.terminate(); print('Stopped.')